In [61]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


### Define helping functions

In [62]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [63]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [64]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


### 1. Load data and investigate the structe

In [65]:
df=load_data("ALF01.csv")

✓ Successfully loaded dataset with 5040 rows and 14 columns


,STATISTIC,Statistic Label,TLIST(A1),Year,C02076V02508,Age Group,C02199V02655,Sex,C04283V05060,Education Attainment Level,C03788V04538,NUTS 2 Region,UNIT,VALUE
0,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),IE0,Ireland,%,74.9
1,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),811D29A4-1E05-4F4A-B61B-43DE64E07B29,Northern and Western,%,74.0
2,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),97227605-23D5-4B98-BDB8-3EB5D0682A5A,Southern,%,72.8
3,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),C32761EB-7A21-4B95-8C7A-DB195C2955CE,Eastern and Midland,%,76.6
4,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,10,Less than primary (Level 0),IE0,Ireland,%,NaN


In [66]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 5,040 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 14 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(A1)', 'Year', 'C02076V02508', 'Age Group', 'C02199V02655', 'Sex', 'C04283V05060', 'Education Attainment Level', 'C03788V04538', 'NUTS 2 Region', 'UNIT', 'VALUE']


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5040 entries, 0 to 5039
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   STATISTIC                   5040 non-null   object 
 1   Statistic Label             5040 non-null   object 
 2   TLIST(A1)                   5040 non-null   int64  
 3   Year                        5040 non-null   int64  
 4   C02076V02508                5040 non-null   object 
 5   Age Group                   5040 non-null   object 
 6   C02199V02655                5040 non-null   object 
 7   Sex                         5040 non-null   object 
 8   C04283V05060                5040 non-null   object 
 9   Education Attainment Level  5040 non-null   object 
 10  C03788V04538                5040 non-null   object 
 11  NUTS 2 Region               5040 non-null   object 
 12  UNIT                        5040 non-null   object 
 13  VALUE                       3037 

Value (employment rate) and year are quantitative. 
TLIST(A1) is the encode of year and is also quantitative.
Others are nominal.

### 2. Invest each column and data distribution

In [68]:

# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['Year']=df['Year'].astype('category')
df['TLIST(A1)']=df['TLIST(A1)'].astype('category')

# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 13


In [69]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,3037.0,74.34623,13.911852,24.3,66.0,76.9,85.2,100.0,39.742063,578


There are nearly 40% percent missing values in the column value (employment rate). Needed to be cleaned later.

In [70]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,5040,1,ALF01C01,5040,0.0,1
Statistic Label,5040,1,Annual employment rate,5040,0.0,1
TLIST(A1),5040,7,2019,720,0.0,7
Year,5040,7,2019,720,0.0,7
C02076V02508,5040,10,-,504,0.0,10
Age Group,5040,10,20 - 24 years,504,0.0,10
C02199V02655,5040,3,-,1680,0.0,3
Sex,5040,3,Both sexes,1680,0.0,3
C04283V05060,5040,6,-,840,0.0,6
Education Attainment Level,5040,6,Less than primary (Level 0),840,0.0,6


According to the table,age groups has the most categories, 10 in total. It would affect the visualization design. <br>
Year comes second but it is actually quantitative. <br>
 Education Level has 6 labels, and may also need to be carefully considered when encoding.<br>
 The number of categories of other features are fine.

In [71]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,ALF01C01,5040



Statistic Label


,value,count
0,Annual employment rate,5040



TLIST(A1)


...

,value,count
0,811D29A4-1E05-4F4A-B61B-43DE64E07B29,1260
1,97227605-23D5-4B98-BDB8-3EB5D0682A5A,1260
2,C32761EB-7A21-4B95-8C7A-DB195C2955CE,1260
3,IE0,1260



NUTS 2 Region


,value,count
0,Eastern and Midland,1260
1,Ireland,1260
2,Northern and Western,1260
3,Southern,1260



UNIT


,value,count
0,%,5040


In [72]:
# Count the missing rows in every attributes.
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


STATISTIC


,total_rows,valid_rows,missing_rows,missing_pct
STATISTIC,,,,
ALF01C01,5040,3037,2003,39.74



Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Annual employment rate,5040,3037,2003,39.74



TLIST(A1)


...

,total_rows,valid_rows,missing_rows,missing_pct
C03788V04538,,,,
811D29A4-1E05-4F4A-B61B-43DE64E07B29,1260,635,625,49.60
97227605-23D5-4B98-BDB8-3EB5D0682A5A,1260,760,500,39.68
C32761EB-7A21-4B95-8C7A-DB195C2955CE,1260,791,469,37.22
IE0,1260,851,409,32.46



NUTS 2 Region


,total_rows,valid_rows,missing_rows,missing_pct
NUTS 2 Region,,,,
Northern and Western,1260,635,625,49.60
Southern,1260,760,500,39.68
Eastern and Midland,1260,791,469,37.22
Ireland,1260,851,409,32.46



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
%,5040,3037,2003,39.74


### 3. Clean the data
Drop the useless colomns and rows with missing values.

Some columns are the encoding of other columns, including STATISTIC, TLIST(A1), C02076V02508, C02199V02655, C04283V05060, C03788V04538. These columns should be deleted. <br>
Besides, Statistic Label and UNIT have only one value which indicate the sematic meaning of the data. They are not useful and can be deleted.

In [73]:
# Drop encoded columns and constant metadata that are not useful for visualization.
col_to_drop = [
    "STATISTIC", "TLIST(A1)", "C02076V02508", "C02199V02655",
    "C04283V05060", "C03788V04538", "Statistic Label", "UNIT"
]
df = df.drop(columns=col_to_drop)

print("Columns dropped successfully: ")
print(col_to_drop)

Columns dropped successfully: 
['STATISTIC', 'TLIST(A1)', 'C02076V02508', 'C02199V02655', 'C04283V05060', 'C03788V04538', 'Statistic Label', 'UNIT']


In [74]:
category_columns = df.select_dtypes('category').columns
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
Year,5040,7,2019,720,0.0,7
Age Group,5040,10,20 - 24 years,504,0.0,10
Sex,5040,3,Both sexes,1680,0.0,3
Education Attainment Level,5040,6,Less than primary (Level 0),840,0.0,6
NUTS 2 Region,5040,4,Eastern and Midland,1260,0.0,4


Rows with missing value should be deleted

In [75]:
# Remove rows with missing values in any column.
df = df.dropna()

print("Rows with missing values removed.")

Rows with missing values removed.


In Education Level, the not-stated does not represent a defined educational attainment category and cannot be meaningfully compared with other category. It should be removed.

In [76]:
# Remove rows with an unspecified educational attainment level.
col = "Education Attainment Level"
df = df.loc[df[col] != "Level of education - not stated"].copy()

print("Education Attainment Level removed")

Education Attainment Level removed


### 4. Change the labels into more readable labels for further visualization
VALUE -> Employment Rate
Education Attainment Level -> Education Level <br>
NUTS 2 Region -> Region <br>
Levels of Education (Levels 0-8) -> All levels<br>
Tertiary education (Levels 5-8)	-> Tertiary (5–8)<br>
Upper secondary and post-secondary non-tertiar...	-> Upper / Post-Secondary (3–4)<br>
Primary and lower secondary education Levels ...	-> Primary / Lower Secondary (1–2)<br>
15-24 years -> 15-24

In [77]:
# Rename columns to make visualization labels easier to read.
df = df.rename(columns={
    "VALUE":"Employment Rate",
    "Education Attainment Level": "Education Level",
    "NUTS 2 Region": "Region",
})

# Shorten education labels while preserving the categorical data type.
labels = {
    "Levels of Education (Levels  0-8)": "All levels",
    "Tertiary education (Levels 5-8)": "Tertiary (5-8)",
    "Upper secondary and post-secondary non-tertiary education (Levels 3 and 4)": "Upper / Post-Secondary (3–4)",
    "Primary and lower secondary education (Levels 1-2)": "Primary / Lower Secondary (1–2)",
}
df["Education Level"] = df["Education Level"].cat.rename_categories(labels)

# Remove the trailing "years" from age labels and preserve categorical values.
df["Age Group"] = df["Age Group"].cat.rename_categories(
    lambda label: label.removesuffix("years").rstrip()
)

# Refresh the column list for subsequent categorical summaries.
category_columns = df.select_dtypes("category").columns
display(df.head())


,Year,Age Group,Sex,Education Level,Region,Employment Rate
0,2019,All ages,Both sexes,All levels,Ireland,74.9
1,2019,All ages,Both sexes,All levels,Northern and Western,74.0
2,2019,All ages,Both sexes,All levels,Southern,72.8
3,2019,All ages,Both sexes,All levels,Eastern and Midland,76.6
8,2019,All ages,Both sexes,Primary / Lower Secondary (1–2),Ireland,52.8


In [78]:
# Remove the unused category label from subsequent frequency tables.
for col in df.select_dtypes("category").columns:
    df[col] = df[col].cat.remove_unused_categories()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2838 entries, 0 to 5035
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Year             2838 non-null   category
 1   Age Group        2838 non-null   category
 2   Sex              2838 non-null   category
 3   Education Level  2838 non-null   category
 4   Region           2838 non-null   category
 5   Employment Rate  2838 non-null   float64 
dtypes: category(5), float64(1)
memory usage: 58.4 KB


In [79]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


Year


,value,count
0,2019,464
1,2020,455
2,2023,395
3,2021,388
4,2022,385
5,2024,376
6,2025,375



Age Group


,value,count
0,All ages,333
1,55 - 59,318
2,60 - 64,310
3,50 - 54,306
4,45 - 49,288
5,40 - 44,283
6,35 - 39,270
7,30 - 34,249
8,20 - 24,245
9,25 - 29,236



Sex


,value,count
0,Both sexes,1003
1,Male,954
2,Female,881



Education Level


,value,count
0,All levels,840
1,Tertiary (5-8),809
2,Upper / Post-Secondary (3–4),783
3,Primary / Lower Secondary (1–2),406



Region


,value,count
0,Ireland,771
1,Eastern and Midland,735
2,Southern,713
3,Northern and Western,619


### 4. Save the data

In [80]:
# Save the cleaned data without the DataFrame index.
df.to_csv("AnnualEmploymentRate.csv", index=False)